# blackjack-counter — M4 detector fine-tune (Colab, free T4)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Inputs: `data/dataset_yolo.zip` built locally by `scripts/export_dataset.py`
(synthetic frames = train; real captures = val/test).

Run every cell top to bottom. The last cell downloads `bjcounter_weights.zip` —
unzip its contents into `models/` in the repo.

Uploads leave this machine ONLY here (labeled card frames to Colab); the runtime
app itself makes zero network calls (BUILD-GUIDE security rule §4.2).

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics==8.4.86
import ultralytics
ultralytics.checks()

In [ ]:
# Upload dataset_yolo.zip (~250 MB; tip: for flaky connections, put the zip in
# Google Drive instead, mount it, and point ZIP_PATH at it).
from pathlib import Path

if not Path('/content/dataset/data.yaml').exists():
    from google.colab import files
    uploaded = files.upload()  # choose dataset_yolo.zip
    zip_name = next(iter(uploaded))
    !unzip -q "{zip_name}" -d /content

assert Path('/content/dataset/data.yaml').exists(), 'dataset not in place'
for split in ('train', 'val', 'test'):
    n = len(list(Path(f'/content/dataset/images/{split}').glob('*.png')))
    print(f'{split}: {n} frames')

In [ ]:
# Fine-tune YOLOv8n. Geometry-preserving augments only: card corners are
# orientation-specific (top-left glyph), so flips/rotations would teach the model
# cards that never occur in the trainer. imgsz=1280 keeps 67-84px cards near-native.
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained base; provenance recorded in the last cell
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=1280,
    batch=8,
    patience=15,
    fliplr=0.0, flipud=0.0, degrees=0.0, shear=0.0, perspective=0.0,
    mosaic=0.5, scale=0.2, hsv_h=0.01, hsv_s=0.2, hsv_v=0.2,
    name='bjcounter',
)

In [ ]:
# Held-out evaluation on the REAL test frames (the number that matters).
metrics = model.val(data='/content/dataset/data.yaml', split='test', imgsz=1280)
print(f'test mAP50      = {metrics.box.map50:.4f}   (gate: >= 0.99)')
print(f'test mAP50-95   = {metrics.box.map:.4f}')
print(f'test mean recall= {metrics.box.mr:.4f}   (per-rank gate checked locally)')

In [ ]:
# Export ONNX at 1280 (+ a 960 fallback in case local CPU latency misses budget),
# record provenance hashes + versions, bundle everything for download.
import hashlib, json, shutil
from datetime import date
from pathlib import Path
import torch

out = Path('/content/export_out'); out.mkdir(exist_ok=True)
best_pt = Path(model.trainer.best)
shutil.copy(best_pt, out / 'best.pt')
# opset pinned per PRD risk table / stack-notes §1: stays compatible with the
# locally-pinned onnxruntime==1.27.0 regardless of Colab's onnx package version.
onnx_1280 = YOLO(best_pt).export(format='onnx', imgsz=1280, opset=17)
shutil.copy(onnx_1280, out / 'best.onnx')
onnx_960 = YOLO(best_pt).export(format='onnx', imgsz=960, opset=17)
shutil.copy(onnx_960, out / 'best_960.onnx')

sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
provenance = {
    'date': str(date.today()),
    'base_checkpoint': 'yolov8n.pt (auto-downloaded by ultralytics from github.com/ultralytics/assets releases)',
    'ultralytics': ultralytics.__version__,
    'torch': torch.__version__,
    'onnx_opset': 17,
    'train_imgsz': 1280, 'epochs': 50,
    'colab_test_map50': float(metrics.box.map50),
    'colab_test_map50_95': float(metrics.box.map),
    'colab_test_mean_recall': float(metrics.box.mr),
    'sha256': {
        'yolov8n.pt': sha('yolov8n.pt'),
        'best.pt': sha(out / 'best.pt'),
        'best.onnx': sha(out / 'best.onnx'),
        'best_960.onnx': sha(out / 'best_960.onnx'),
    },
}
(out / 'metrics_colab.json').write_text(json.dumps(provenance, indent=2))
print(json.dumps(provenance, indent=2))

In [ ]:
shutil.make_archive('/content/bjcounter_weights', 'zip', out)
from google.colab import files
files.download('/content/bjcounter_weights.zip')

## After downloading

Unzip `bjcounter_weights.zip` into the repo's `models/` directory so it contains
`best.pt`, `best.onnx`, `best_960.onnx`, `metrics_colab.json` — then run locally:

```
python scripts/eval_detector.py
```

which checks the M4 gates (mAP50 ≥ 0.99, per-rank recall ≥ 0.995, < 100 ms/frame
CPU) with the project's own ONNX pre/post-processing and writes `models/metrics.json`.